In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])

In [ ]:
%%capture
!pip install -q transformers accelerate datasets peft trl bitsandbytes sacrebleu huggingface_hub

In [ ]:
import torch, transformers, peft, trl, platform

print("=" * 55)
print("ENVIRONMENT")
print("=" * 55)
print(f"Python       : {platform.python_version()}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"PEFT         : {peft.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU 0        : {torch.cuda.get_device_name(0)} ({mem:.1f} GB)")
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Active device: {DEVICE}")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token, add_to_git_credential=False)
    print("HuggingFace auth successful.")
except Exception as e:
    print(f"Auth failed: {e}")

In [ ]:
MODEL_ID   = "google/gemma-3-1b-it"
OUTPUT_DIR = "/kaggle/working/stage2_sft_adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Scan for CPT adapter — prints exactly what's mounted
print("Scanning /kaggle/input for CPT adapter...")
CPT_ADAPTER_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "adapter_config.json" in files:
        print(f"  Found adapter at: {root}")
        if "checkpoint-1000" in root:
            CPT_ADAPTER_PATH = root
        elif CPT_ADAPTER_PATH is None:
            CPT_ADAPTER_PATH = root

if CPT_ADAPTER_PATH:
    print(f"\nUsing CPT adapter: {CPT_ADAPTER_PATH}")
else:
    print("\nWARNING: No CPT adapter found — training from base model.")
    print("To fix: Add Data → Your Work → stage1_cpt in the Kaggle sidebar.")

# Hyperparameters
MAX_SEQ_LEN   = 512
LEARNING_RATE = 2e-4
LR_SCHEDULER  = "cosine"
NUM_EPOCHS    = 1
BATCH_SIZE    = 2
GRAD_ACCUM    = 8       # effective batch = 16
WARMUP_STEPS  = 50
WEIGHT_DECAY  = 0.01
MAX_GRAD_NORM = 1.0
SAVE_STEPS    = 300
LOGGING_STEPS = 25
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
MIN_PROMPTS   = 5_000   # hard minimum — fail fast if data loading is broken

print(f"\nMin required prompts : {MIN_PROMPTS:,}")
print(f"Max sequence length  : {MAX_SEQ_LEN}")
print(f"Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f"Tokenizer loaded. Vocab: {tokenizer.vocab_size:,}")

In [ ]:
def make_translation_prompt(english: str, swahili: str) -> str:
    return (
        f"<start_of_turn>user\n"
        f"Translate the following English sentence to Swahili.\n"
        f"English: {english.strip()}\n"
        f"Swahili:<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{swahili.strip()}<end_of_turn>"
    )

def make_instruction_prompt(instruction: str, response: str, input_text: str = "") -> str:
    user_part = instruction.strip()
    if input_text and input_text.strip():
        user_part += f"\n{input_text.strip()}"
    return (
        f"<start_of_turn>user\n"
        f"{user_part}<end_of_turn>\n"
        f"<start_of_turn>model\n"
        f"{response.strip()}<end_of_turn>"
    )

sample = make_translation_prompt("The children are playing.", "Watoto wanacheza.")
print("Sample prompt:")
print(sample)
print(f"Tokens: {len(tokenizer.encode(sample))}")

## Dataset loading — all column names verified from actual output

In [ ]:
from datasets import load_dataset, Dataset
import random

all_prompts = []

print("Loading Rogendo/English-Swahili-Sentence-Pairs...")
try:
    rogendo = load_dataset(
        "Rogendo/English-Swahili-Sentence-Pairs",
        split="train[:6000]"
    )
    print(f"  Columns: {rogendo.column_names}")
    before = len(all_prompts)
    for row in rogendo:
        # CONFIRMED column names from actual run
        en = row.get("English sentence", "")
        sw = row.get("Swahili Translation", "")
        if en and sw and len(str(en)) > 10 and len(str(sw)) > 10:
            all_prompts.append(make_translation_prompt(str(en), str(sw)))
    print(f"  Added: {len(all_prompts) - before:,} pairs")
except Exception as e:
    print(f"  FAILED: {e}")

In [ ]:
print("Loading Sunbird/salt (text-all)...")
try:
    salt = load_dataset("Sunbird/salt", "text-all", split="train[:3000]")
    print(f"  Columns: {salt.column_names}")
    before = len(all_prompts)
    for row in salt:
        # CONFIRMED column names from actual run
        en = row.get("eng_source_text", "")
        sw = row.get("swa_text", "")
        if en and sw and len(str(en)) > 5 and len(str(sw)) > 5:
            all_prompts.append(make_translation_prompt(str(en), str(sw)))
    print(f"  Added: {len(all_prompts) - before:,} pairs")
except Exception as e:
    print(f"  FAILED: {e}")

In [ ]:
print("Loading Svngoku/Inkuba-Swahili-MMT...")
try:
    inkuba_mmt = load_dataset(
        "Svngoku/Inkuba-Swahili-MMT",
        split="train[:4000]"
    )
    print(f"  Columns: {inkuba_mmt.column_names}")
    before = len(all_prompts)
    for row in inkuba_mmt:
        # CONFIRMED: inputs=Swahili, targets=English (SW→EN dataset)
        # We swap: use targets as EN input, inputs as SW output
        en = row.get("targets", "")   # English
        sw = row.get("inputs", "")    # Swahili
        if en and sw and len(str(en)) > 10 and len(str(sw)) > 10:
            all_prompts.append(make_translation_prompt(str(en), str(sw)))
    print(f"  Added: {len(all_prompts) - before:,} pairs")
except Exception as e:
    print(f"  FAILED: {e}")

In [ ]:
print("Loading CohereForAI/aya_dataset (Swahili rows)...")
try:
    aya = load_dataset("CohereForAI/aya_dataset", split="train")
    aya_sw = aya.filter(
        lambda x: x.get("language", "").lower() in ["swahili", "sw"]
    )
    print(f"  Swahili rows found: {len(aya_sw):,}")
    before = len(all_prompts)
    for row in aya_sw:
        inp = row.get("inputs", row.get("input", ""))
        out = row.get("targets", row.get("target", ""))
        if inp and out and len(str(out)) > 10:
            all_prompts.append(make_instruction_prompt(str(inp), str(out)))
    print(f"  Added: {len(all_prompts) - before:,} instruction pairs")
except Exception as e:
    print(f"  FAILED: {e}")

In [ ]:
print("Loading iamshnoo/alpaca-cleaned-swahili...")
try:
    alpaca = load_dataset(
        "iamshnoo/alpaca-cleaned-swahili",
        split="train[:4000]"
    )
    print(f"  Columns: {alpaca.column_names}")
    before = len(all_prompts)
    for row in alpaca:
        # CONFIRMED column names from actual run
        instruction = row.get("instruction", "")
        inp         = row.get("input", "")
        out         = row.get("output", "")
        if instruction and out and len(str(out)) > 10:
            all_prompts.append(make_instruction_prompt(str(instruction), str(out), str(inp)))
    print(f"  Added: {len(all_prompts) - before:,} instruction pairs")
except Exception as e:
    print(f"  FAILED: {e}")
    try:
        print("  Trying saillab/alpaca-swahili-cleaned fallback...")
        alpaca = load_dataset("saillab/alpaca-swahili-cleaned", split="train[:4000]")
        before = len(all_prompts)
        for row in alpaca:
            instruction = row.get("instruction", "")
            inp         = row.get("input", "")
            out         = row.get("output", "")
            if instruction and out and len(str(out)) > 10:
                all_prompts.append(make_instruction_prompt(str(instruction), str(out), str(inp)))
        print(f"  Fallback added: {len(all_prompts) - before:,} pairs")
    except Exception as e2:
        print(f"  Fallback FAILED: {e2}")

In [ ]:
print("Loading lighteval/KenSwQuAD...")
try:
    # CONFIRMED: no train split — only 'test' and 'validation'
    kenswquad = load_dataset("lighteval/KenSwQuAD", split="test[:2000]")
    print(f"  Columns: {kenswquad.column_names}")
    before = len(all_prompts)
    for row in kenswquad:
        question = row.get("question", "")
        context  = row.get("context", "")
        answers  = row.get("answers", row.get("answer", {}))
        if isinstance(answers, dict):
            answer_list = answers.get("text", [])
            answer = answer_list[0] if answer_list else ""
        elif isinstance(answers, list):
            answer = answers[0] if answers else ""
        else:
            answer = str(answers)
        if question and answer and len(str(answer)) > 5:
            all_prompts.append(make_instruction_prompt(
                str(question), str(answer), str(context)
            ))
    print(f"  Added: {len(all_prompts) - before:,} QA pairs")
except Exception as e:
    print(f"  FAILED: {e}")

print(f"\n{'='*50}")
print(f"TOTAL PROMPTS COLLECTED: {len(all_prompts):,}")
print(f"{'='*50}")
print(f"Expected breakdown:")
print(f"  Rogendo    : ~5,800 EN-SW pairs")
print(f"  SALT       : ~2,800 EN-SW pairs")
print(f"  Inkuba MMT : ~3,800 EN-SW pairs (swapped)")
print(f"  Aya        :   ~357 instruction pairs")
print(f"  Alpaca     : ~3,957 instruction pairs")
print(f"  KenSwQuAD  : ~1,500 QA pairs")
print(f"  Total      : ~18,200")

In [ ]:
MIN_PROMPTS = 5_000
if len(all_prompts) < MIN_PROMPTS:
    raise RuntimeError(
        f"INSUFFICIENT DATA: Only {len(all_prompts):,} prompts collected. "
        f"Need at least {MIN_PROMPTS:,}. "
        f"Check which datasets FAILED above and fix before continuing."
    )
print(f"Data gate passed: {len(all_prompts):,} prompts >= {MIN_PROMPTS:,} minimum.")

In [ ]:
random.seed(42)
random.shuffle(all_prompts)

filtered = []
for p in all_prompts:
    token_len = len(tokenizer.encode(p, add_special_tokens=False))
    if 20 < token_len <= MAX_SEQ_LEN:
        filtered.append(p)

steps     = len(filtered) // (BATCH_SIZE * GRAD_ACCUM)
est_hours = steps * 35 / 3600

print(f"After filtering        : {len(filtered):,} prompts")
print(f"Dropped (length)       : {len(all_prompts) - len(filtered):,}")
print(f"Training steps         : {steps:,}")
print(f"Estimated time         : {est_hours:.1f} hours @ 35s/step")

# Auto-trim if over time budget
if est_hours > 10.5:
    max_samples = int(10.5 * 3600 / 35) * (BATCH_SIZE * GRAD_ACCUM)
    filtered = filtered[:max_samples]
    steps = len(filtered) // (BATCH_SIZE * GRAD_ACCUM)
    print(f"\nTrimmed to {len(filtered):,} samples to fit 12h limit")
    print(f"Revised steps: {steps:,} (~{steps * 35 / 3600:.1f} hours)")

print("\nSample prompt:")
print(filtered[0])

## Tokenize

In [ ]:
raw_dataset = Dataset.from_dict({"text": filtered})

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
        return_attention_mask=True,
    )

print("Tokenising...")
tokenised = raw_dataset.map(
    tokenize, batched=True, batch_size=500,
    remove_columns=["text"], desc="Tokenising",
)

split = tokenised.train_test_split(test_size=0.02, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]
print(f"Train rows : {len(train_dataset):,}")
print(f"Eval rows  : {len(eval_dataset):,} (unused)")

## Load base model + CPT adapter

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading base model: {MODEL_ID}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    dtype=torch.bfloat16,
    attn_implementation="eager",
)

if CPT_ADAPTER_PATH:
    print(f"Loading CPT adapter: {CPT_ADAPTER_PATH}")
    model = PeftModel.from_pretrained(model, CPT_ADAPTER_PATH, is_trainable=False)
    model = model.merge_and_unload()
    print("CPT adapter merged into base model.")
else:
    print("No CPT adapter — training from base model.")

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model.enable_input_require_grads()
print(f"VRAM after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    inference_mode=False,
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,} ({100*trainable/total:.2f}%)")
print(f"Total params     : {total:,}")

## Train

In [ ]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling, Trainer
import time

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    fp16=False,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    eval_strategy="no",
    load_best_model_at_end=False,
    save_total_limit=3,
    report_to="none",
    dataloader_num_workers=2,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
    dataloader_pin_memory=False,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

steps = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print(f"Starting SFT training...")
print(f"Total steps    : {steps:,}")
print(f"Estimated time : ~{steps * 35 / 3600:.1f} hours @ 35s/step")
print(f"First save at  : step {SAVE_STEPS} (~{SAVE_STEPS * 35 / 3600:.1f}h)")
print()

start = time.time()
train_result = trainer.train()
elapsed = time.time() - start

print(f"\nTraining complete in {elapsed/3600:.2f} hours.")
print(f"Final train loss : {train_result.training_loss:.4f}")

## Save adapter and metrics

In [ ]:
import json

adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

metrics = {
    "stage"          : 2,
    "phase"          : "SFT_v3",
    "base_model"     : MODEL_ID,
    "cpt_adapter"    : CPT_ADAPTER_PATH,
    "train_loss"     : round(train_result.training_loss, 4),
    "train_samples"  : len(train_dataset),
    "total_prompts"  : len(filtered),
    "learning_rate"  : LEARNING_RATE,
    "max_seq_len"    : MAX_SEQ_LEN,
    "elapsed_hours"  : round(elapsed/3600, 2),
}
with open(f"{OUTPUT_DIR}/stage2_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Adapter saved : {adapter_path}")
print(f"Metrics saved : {OUTPUT_DIR}/stage2_metrics.json")

## Smoke test

In [ ]:
model.eval()

GEN_CONFIG = {
    "temperature": 0.3, "top_p": 0.95, "top_k": 64,
    "max_new_tokens": 128, "repetition_penalty": 1.1, "do_sample": True,
}

def translate(english: str) -> str:
    prompt = (
        f"<start_of_turn>user\n"
        f"Translate the following English sentence to Swahili.\n"
        f"English: {english}\n"
        f"Swahili:<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs, **GEN_CONFIG,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

TEST_PAIRS = [
    ("The students are learning at school today.",
     "Wanafunzi wanasoma shuleni leo."),
    ("She went to the market to buy vegetables and fruit.",
     "Alienda sokoni kununua mboga na matunda."),
    ("The government announced new policies for healthcare.",
     "Serikali ilitangaza sera mpya za huduma za afya."),
    ("Climate change is affecting agriculture across Africa.",
     "Mabadiliko ya tabianchi yanaathiri kilimo katika Afrika yote."),
    ("The beautiful chairs fell on the ground.",
     "Viti vizuri vilianguka chini."),  # noun-class agreement test
]

print("POST-SFT SMOKE TEST")
print("=" * 65)
for en, ref in TEST_PAIRS:
    hyp = translate(en)
    print(f"EN : {en}")
    print(f"REF: {ref}")
    print(f"HYP: {hyp}")
    print()

## Mini FLORES-200 eval (100 sentences)

In [ ]:
from sacrebleu.metrics import BLEU, CHRF

print("Loading FLORES-200 (100 sentences)...")
flores_en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")
flores_sw = load_dataset("openlanguagedata/flores_plus", "swh_Latn", split="devtest")

MINI_EVAL = 100
en_sents = flores_en["text"][:MINI_EVAL]
sw_refs  = flores_sw["text"][:MINI_EVAL]

hypotheses = []
start = time.time()
for i, en in enumerate(en_sents):
    hypotheses.append(translate(en))
    if (i+1) % 25 == 0:
        print(f"  [{i+1}/{MINI_EVAL}] {time.time()-start:.0f}s elapsed")

bleu = BLEU(effective_order=True).corpus_score(hypotheses, [sw_refs]).score
chrf = CHRF(word_order=2).corpus_score(hypotheses, [sw_refs]).score

print(f"\n{'='*55}")
print(f"MINI FLORES-200 RESULTS (100 sentences)")
print(f"{'='*55}")
print(f"BLEU   : {bleu:.2f}  (target > 27.6)")
print(f"chrF++ : {chrf:.2f}  (target > 56.8)")
print(f"Stage 0 base model     : ~0 BLEU (Finnish output)")
print(f"Swahili Gemma 1B target: 27.6 BLEU / 56.8 chrF++")

metrics.update({
    "mini_flores_bleu": round(bleu, 4),
    "mini_flores_chrf": round(chrf, 4),
    "mini_eval_size"  : MINI_EVAL,
})
with open(f"{OUTPUT_DIR}/stage2_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nFinal metrics saved.")
print("Stage 2 SFT v3 complete. Download stage2_sft_adapter/ from output panel.")